In [ ]:
import anndata as ad
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import warnings
import os  
import loompy
import gzip
import shutil
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import mannwhitneyu

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(dpi=50, dpi_save=300, format='png', 
                             frameon=False, transparent=True, fontsize=10, figsize=(4, 4))

warnings.simplefilter(action='ignore', category=FutureWarning)

plt.rcParams["image.aspect"] = "equal"
plt.rcParams["figure.figsize"] = ([4, 4])  
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.grid'] = False

colorrs = ["#4DBBD5", "#00A087", "#E64B35","#3C5488", "#F39B7F", "#8491B4",
        "#91D1C2",  "#B9C984",  "#9ACBDE", "#F494BE", "#EDCAE0", 
        "#C8CADF", "#F47892", "#F6A395",  "#C9AFA2", "#ABADC5", "#AEB9AC", 
        "#4b6aa8", "#3ca0cf", "#c376a7", "#ad98c3", "#cea5c7",
        "#53738c", "#a5a9b0", "#a78982", "#696a6c", "#92699e",
        "#d69971", "#df5734", "#6c408e", "#ac6894", "#d4c2db",
        "#537eb7", "#83ab8e", "#ece399", "#405993", "#cc7f73",
        "#b95055", "#d5bb72", "#bc9a7f", "#e0cfda", "#d8a0c0",
        "#d69a55", "#64a776", "#cbdaa9",
        "#efd2c9", "#da6f6d", "#ebb1a4", "#a44e89", "#a9c2cb",
        "#b85292", "#6d6fa0", "#8d689d", "#c8c7e1", "#d25774",
        "#c49abc", "#927c9a", "#3674a2", "#9f8d89", "#72567a",
        "#63a3b8", "#c4daec", "#61bada", "#b7deea", "#e29eaf",
        "#4490c4", "#e6e2a3",  "#c4612f", "#9a70a8",
        "#76a2be", "#408444", "#c6adb0", "#9d3b62", "#2d3462"]

In [ ]:
raw_adata = adata.raw.to_adata()
CD8T = raw_adata[raw_adata.obs['celltype_major2']=='CD8+ T cells'].copy()

In [ ]:
markers = ["CD8A","CD8B","CCR7","LEF1","GPR183","GZMH","GZMA","GZMM","GZMK","MKI67","SLC4A10","TRAV1-2","TRDV2","TRGV9"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#4DBBD5","white","#E64B35"], N=30)

with plt.rc_context({"axes.linewidth":0.5,"xtick.major.width":0.5,"ytick.major.width":0.5}):
    fig = sc.pl.stacked_violin(CD8T, markers, groupby="celltype", cmap=cmap, figsize=(7,2.5),
                               linewidth=0.5, colorbar_title="Expression", swap_axes=False, return_fig=True)

fig.savefig("Fig.9/CD8_stacked_violin_markers.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
signature_dict = {
    "Naive": ["CCR7","TCF7","LEF1","SELL"],
    "Activation_Effector": ["FAS","CD44","CD69","CD38","NKG7","KLRB1","KLRD1","KLRG1","CX3CR1","CD300A","FGFBP2","ID2","ID3","PRDM1","RUNX3","TBX21","ZEB2","BATF","NR4A1","NR4A2","HOPX","FOS","FOSB","FOSL2","JUN","JUNB","JUND","STAT1","STAT3","EOMES","AHR"],
    "Cytotoxic": ["PRF1","IFNG","GNLY","NKG7","GZMA","GZMB","GZMH","GZMK","GZMM","KLRK1","KLRB1","KLRD1","FCGR3A","FGFBP2","ZEB2","CTSW","CST7"],
    "Exhaustion": ["HAVCR2","CXCL13","CCL3","SIRPG","IFNG","TIGIT","GZMB","PDCD1","PARK7","TNFRSF9","ACP5","CTLA4","RBPJ","MIR155","CXCR6","CD27","FKBP1A","BST2","TPI1","MIR155HG","PTTG1","CD63","SAMSN1","RGS1","CD27-AS1","ITGAE","MIR4632","HLA-DRA","IGFLR1","KRT86","ENTPD1","DUSP4","SIT1","TOX","PHLDA1","CCND2","GPR25","LAYN","PRDX5","SARDH","FASLG","MIR3917","ANXA5","CTSD","PDIA6","RANBP1","FKBP1A-SDCBP2","COTL1","TNFRSF1B","IDH2","CD38","CD82","LAG3","MIR497HG","APOBEC3C","ITM2A","COX5A","IFI35","NDFIP2","TNFRSF18","KRT81","DNPH1","RGS2","HMGN1","DYNLL1","SNRPB","STRA13","SYNGR2","RAB27A","PSMC3","GALM","FABP5","UBE2L6","MYO7A","PRDX3","DDIT4","STMN1","CDK2AP2","VCAM1","SNAP47","PSMB3","ISG15","HLA-DRB5","CKS2","TNIP3","CD7","PSMD4","ATP6V1C2","PSMD8","HLA-DRB6"],
    "TCR_Signaling": ["CALM1","CALM2","CALM3","CD4","CAST","CD247","CD3D","CD3E","CD3G","CSK","DOK2","FYN","LCK","NFATC1","NFATC2","PLEK","PTPN11","PTPN13","PTPN2","PTPN22","PTPN4","PTPN6","PTPN7","PTPRC","PTPRCAP","S100A10","S100A11","S100A4","S100A6","ZAP70","DUSP1","DUSP2","DUSP4","DUSP16","LAT","FOS","FOSB","FOSL2","JUN","JUNB","JUND","NR4A1","NR4A2","BATF","IRF1","SH2D1A","SH2D2A","MAP2K3","MAP3K4","MAP3K8","MAP4K1","NFKB2","NFKBIA","NFKBIZ","REL","RELB"],
    "IFN response": [ 'SOCS3', 'SOCS1', 'IFNG', 'STAT1', 'SLC7A5', 'DDIT3','ZC3H12A', 'CD74', 'CCL5', 'GAPDH', 'IRF1','CCL5', 'B2M', 'GAPDH'],
    "Unhelped CD4 T": ["LRRC61","KNTC1","ARSB","KLF12","CMKLR1","RBM38","SH3TC1","CLSPN","PRR5L","KIAA1671","STK32C","ZMIZ1","IFITM1","IFITM2","IFITM3","PRR5","CKAP4","STMN1","ITGAM","CCNF","CHN2","ENTPD1","VKORC1L1","P2RX7","OSBPL3","UBE2C","SPECC1","ZEB2","ST3GAL1","KLRG1","SYTL2","KLRB1","CORO2A","GPR55","S1PR5","EMILIN2","RUNX1","HAVCR2","ANXA1","HECTD2","ARHGAP11A","ARHGAP11B","PHLDB2","NCAPG2","ADAM8","SLC43A3","SUFU","CX3CR1","BHLHE40","WIPI1","KLRC4","KLRC4-KLRK1","KLRC3","KLRC2","KLRC1","ESPL1","MICAL3","IRF4","L1CAM","SEMA4A","CCNB1","TSPAN32","TIAM1","BUB1","MYADM","IL18RAP","IL9R","LDLR","KIAA0513","NFIC","APOBR","BMPR2","CDC6","GZMA"],
    "Apoptosis": ["GZMA","GZMB","GZMH","PRF1","CASP3","TNFSF10","TNFRSF10A","TNFRSF10B","TNFSF12","TNFRSF25","FAS","FASLG","FADD","TRADD","CASP8","IRF1","TP53","XAF1","BCL2L11"]
}

for score_key, gene_list in signature_dict.items():
    genes = [g for g in dict.fromkeys(gene_list) if g in CD8T.var_names]
    if genes: sc.tl.score_genes(CD8T, gene_list=genes, score_name=f"{score_key} scores", random_state=42)

In [ ]:
score_cols = ["Exhaustion scores","Cytotoxic scores"]
pb_df = CD8T.obs.groupby(["Sample","Condition","celltype"], observed=False)[score_cols].mean().dropna().reset_index()
celltypes = pb_df.groupby("celltype", observed=False)["Exhaustion scores"].mean().sort_values(ascending=False).index.tolist()

def kw_test(df, score):
    valid = [ct for ct,g in df.groupby("celltype", observed=False) if g.groupby("Condition").size().reindex(["HC","MKPP","SKPP"], fill_value=0).min() >= 3]
    res = [(ct, *kruskal(*[df.loc[(df["celltype"]==ct)&(df["Condition"]==c), score] for c in ["HC","MKPP","SKPP"]])) for ct in valid]
    out = pd.DataFrame(res, columns=["CellType","H_stat","p_value"])
    if out.empty: return out
    out["p_adj"] = multipletests(out["p_value"], method="bonferroni")[1]
    out["Significance"] = out["p_adj"].apply(lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")
    return out

stats = {"Exhaustion scores":kw_test(pb_df,"Exhaustion scores"), "Cytotoxic scores":kw_test(pb_df,"Cytotoxic scores")}
colors = {"Exhaustion scores":"#E64B35","Cytotoxic scores":"#00A087"}

fig, axes = plt.subplots(2,1, figsize=(max(6,len(celltypes)*0.6),7), sharex=True, gridspec_kw={"hspace":0.1})

for ax, score in zip(axes, score_cols):
    color = colors[score]
    sns.boxplot(data=pb_df, x="celltype", y=score, order=celltypes, ax=ax, width=0.6, showfliers=False,
                boxprops={"facecolor":"none","edgecolor":color,"linewidth":1.5},
                whiskerprops={"color":color,"linewidth":1.5}, capprops={"color":color,"linewidth":1.5},
                medianprops={"color":color,"linewidth":1.5})
    ymin, ymax = pb_df[score].min(), pb_df[score].max(); yrange = ymax-ymin
    sig = stats[score].set_index("CellType")["Significance"] if not stats[score].empty else {}
    for i, ct in enumerate(celltypes):
        star = sig.get(ct,"ns")
        if star != "ns": ax.text(i, ymax+yrange*0.05, star, ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.set_ylim(ymin-yrange*0.05, ymax+yrange*0.2)
    ax.set_ylabel(score.replace("scores","Score"), fontsize=12, fontweight="bold")
    ax.set_xlabel(""); ax.grid(False); ax.set_facecolor("white")
    sns.despine(ax=ax)

axes[0].tick_params(axis="x", length=0)
axes[1].set_xticklabels(celltypes, rotation=90, ha="center", fontsize=11)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scikit_posthocs as sp
from scipy.stats import kruskal

target_celltypes = ["CD8T_04_Effector_GZMA_GZMM","CD8T_05_Effector_GZMK","MAIT","γδT"]
score_col = "Exhaustion scores"
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("MKPP","SKPP"),("HC","SKPP")]
x = np.arange(3)

plot_df = CD8T.obs.loc[CD8T.obs["celltype"].isin(target_celltypes)].groupby(
    ["Sample","Condition","celltype"], observed=True)[score_col].mean().dropna().reset_index()

ymin, ymax = plot_df[score_col].agg(["min","max"]); yrange = ymax-ymin if ymax!=ymin else 1
sig_heights = [ymax+yrange*i for i in [0.18,0.36,0.54]]
fig, axes = plt.subplots(1,len(target_celltypes), figsize=(len(target_celltypes)*2,4.2), sharey=True)

for ax, celltype in zip(axes,target_celltypes):
    sub = plot_df[plot_df["celltype"]==celltype]
    groups = [sub.loc[sub["Condition"]==c,score_col] for c in condition_order]
    kw_p = kruskal(*groups).pvalue
    dunn = sp.posthoc_dunn(sub, val_col=score_col, group_col="Condition", p_adjust="bonferroni")
    pvals = [dunn.loc[a,b] for a,b in pairs]
    stars = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

    for i,(condition,color) in enumerate(zip(condition_order,colors)):
        data = sub.loc[sub["Condition"]==condition,score_col].to_numpy()
        if len(data)==0: continue
        ax.scatter(np.random.normal(x[i]-0.25,0.04,len(data)), data, s=28, color=color, alpha=0.8, edgecolor="white", linewidth=0.3, zorder=3)
        box = ax.boxplot(data, positions=[x[i]-0.05], widths=0.1, patch_artist=True, showfliers=False, zorder=4)
        plt.setp(box["boxes"], facecolor="white", edgecolor=color, linewidth=1.5)
        plt.setp(box["whiskers"]+box["caps"]+box["medians"], color=color, linewidth=1.5)
        if len(data)>=2:
            violin = ax.violinplot(data, positions=[x[i]+0.1], showextrema=False)
            for body in violin["bodies"]:
                body.set(facecolor=color, alpha=0.8, edgecolor="none")
                v = body.get_paths()[0].vertices; v[:,0] = np.clip(v[:,0], x[i]+0.1, np.inf)

    for (a,b),star,h in zip(pairs,stars,sig_heights):
        x1,x2 = condition_order.index(a),condition_order.index(b)
        ax.plot([x1,x2],[h,h], lw=1, color="black")
        ax.text((x1+x2)/2,h,star,ha="center",va="bottom",fontsize=11,fontweight="bold" if star!="ns" else "normal")

    title = "_".join(celltype.split("_")[:3]) if len(celltype.split("_"))>=3 else celltype
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xticks(x, condition_order, rotation=45, ha="right", fontsize=11)
    ax.tick_params(axis="y", labelsize=12)
    ax.spines[["top","right"]].set_visible(False)
    ax.set_ylim(ymin-yrange*0.1, ymax+yrange*0.72)
    ax.grid(False)

axes[0].set_ylabel("Exhaustion scores", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("Fig.4/CD8_subtype_Exhaustion_Raincloud_byCondition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
exhaustion_genes = ["LAG3","PDCD1","HAVCR2","KLRG1","CD200","CD244","CD160","CTLA4","CD80","TIGIT"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

CD8T.obs["celltype_simple"] = CD8T.obs["celltype"].apply(lambda x: "_".join(str(x).split("_")[:3]) if len(str(x).split("_"))>=3 else str(x))

dp = sc.pl.DotPlot(CD8T, var_names=exhaustion_genes, groupby="celltype_simple", standard_scale="var", figsize=(5,2.5))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True)
ax = axes["mainplot_ax"]

if "size_legend_ax" in axes:
    axes["size_legend_ax"].tick_params(labelsize=12)
    axes["size_legend_ax"].set_title("Pct. exp.", fontsize=12)

if "color_legend_ax" in axes:
    axes["color_legend_ax"].tick_params(labelsize=12)
    axes["color_legend_ax"].set_title("Avg. exp.", fontsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

ax.tick_params(axis="x", labelsize=12, rotation=45)
ax.tick_params(axis="y", labelsize=12)
ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig("Fig.4/CD8T_exhaustion_genes_by_celltype.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
cytotoxic_genes = ["PRF1","IFNG","GNLY","NKG7","GZMA","GZMB","GZMH","GZMK","GZMM","KLRK1","KLRB1","KLRD1","FCGR3A","FGFBP2","ZEB2","CTSW","CST7"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(CD8T, var_names=cytotoxic_genes, groupby="Condition", standard_scale="var", figsize=(8,1.5))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True)
ax = axes["mainplot_ax"]

if "size_legend_ax" in axes:
    axes["size_legend_ax"].tick_params(labelsize=12)
    axes["size_legend_ax"].set_title("Pct. exp.", fontsize=12)

if "color_legend_ax" in axes:
    axes["color_legend_ax"].tick_params(labelsize=12)
    axes["color_legend_ax"].set_title("Avg. exp.", fontsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

ax.tick_params(axis="x", labelsize=12, rotation=45)
ax.tick_params(axis="y", labelsize=12)
ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig("Fig.4/CD8T_cytotoxic_genes_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
cytotoxic_genes = ["PRF1","IFNG","GNLY","NKG7","GZMA","GZMB","GZMH","GZMK","GZMM","KLRK1","KLRB1","KLRD1","FCGR3A","FGFBP2","ZEB2","CTSW","CST7"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(CD8T, var_names=cytotoxic_genes, groupby="Condition", standard_scale="var", figsize=(8,1.5))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(size_title="Pct. exp.", colorbar_title="Avg. exp.")
axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key, title in [("size_legend_ax","Pct. exp."), ("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values(): spine.set_linewidth(0.8)
ax.tick_params(axis="x", labelsize=12, rotation=45)
ax.tick_params(axis="y", labelsize=12)
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.4/CD8T_cytotoxic_genes_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
score_order = ["Apoptosis scores","Pyroptosis scores"]
condition_order = ["HC","MKPP","SKPP"]

score_matrix = CD8T.obs.groupby("Condition", observed=False)[score_order].mean().reindex(condition_order).T
score_matrix = score_matrix.apply(lambda x: 2*(x-x.min())/(x.max()-x.min())-1 if x.max()!=x.min() else x*0, axis=1)

cmap = LinearSegmentedColormap.from_list("blue_white_red", ["#4DBBD5B2","white","#E64B35B2"], N=100)
fig, ax = plt.subplots(figsize=(5,1.5))

sns.heatmap(score_matrix, cmap=cmap, vmin=-1, vmax=1, linewidths=0.6, linecolor="white",
            cbar_kws={"label":"Signature score","shrink":0.75,"aspect":15}, ax=ax)

ax.set(xlabel="", ylabel="")
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=15, color="black")
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=15, color="black")

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(colors="black", labelsize=14)
cbar.set_label("Signature score", color="black", fontsize=14)

plt.tight_layout()
plt.savefig("Fig.4/CD8T_PCD_score_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
genes_to_plot = ["GZMA","GZMB","CASP3","TNFSF10","CASP8","FAS","IRF1","XAF1"]
comparison_pairs = [("HC","MKPP"),("HC","SKPP"),("MKPP","SKPP")]

expr = CD8T.raw[:, genes_to_plot].X if CD8T.raw is not None else CD8T[:, genes_to_plot].X
if hasattr(expr, "toarray"): expr = expr.toarray()

exp_df = pd.DataFrame(expr, index=CD8T.obs_names, columns=genes_to_plot)
sample_df = pd.concat([CD8T.obs[["Sample","Condition"]], exp_df], axis=1).groupby(
    ["Sample","Condition"], observed=True)[genes_to_plot].mean().reset_index()

plot_df = sample_df.melt(id_vars=["Sample","Condition"], value_vars=genes_to_plot,
                         var_name="Gene", value_name="Expression")

stars_dict = {}
for gene in genes_to_plot:
    sub = plot_df[plot_df["Gene"]==gene]
    groups = [sub.loc[sub["Condition"]==c,"Expression"] for c in ["HC","MKPP","SKPP"]]
    kw_p = kruskal(*groups).pvalue
    dunn = sp.posthoc_dunn(sub, val_col="Expression", group_col="Condition", p_adjust="bonferroni")
    pvals = [dunn.loc[a,b] for a,b in comparison_pairs]
    stars_dict[gene] = {f"{a}_vs_{b}":"***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
                        for (a,b),p in zip(comparison_pairs,pvals)}

In [ ]:
apoptosis_genes = ["GZMA","GZMB","CASP3","TNFSF10","CASP8","FAS","IRF1","XAF1"]
pyroptosis_genes = ["NLRP3","PYCARD","CASP1","CASP4","CASP5","GSDMD","GSDME","IL18"]
gene_pool = CD8T.raw.var_names if CD8T.raw is not None else CD8T.var_names
groups = [([g for g in apoptosis_genes if g in gene_pool],"#E64B35","Apoptosis"),
          ([g for g in pyroptosis_genes if g in gene_pool],"#4DBBD5","Pyroptosis")]
valid_genes = [g for genes,_,_ in groups for g in genes]

cmap = LinearSegmentedColormap.from_list("blue_white_red", ["#4DBBD5","white","#E64B35"])
dp = sc.pl.DotPlot(CD8T, var_names=valid_genes, groupby="Condition", standard_scale="var",
                   categories_order=["HC","MKPP","SKPP"], figsize=(9,3))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.8, grid=True, smallest_dot=10,
         largest_dot=400, dot_max=0.4).legend(size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]
ax.spines[["left","bottom"]].set_visible(True); ax.spines[["left","bottom"]].set_linewidth(1)
ax.spines[["top","right"]].set_visible(False)
ax.tick_params(which="both", direction="out", length=3, pad=10)
plt.setp(ax.get_xticklabels(), fontsize=13, rotation=90, ha="center", va="top")
plt.setp(ax.get_yticklabels(), fontsize=13)
ax.set(xlabel="", ylabel="")

trans = ax.get_xaxis_transform(); start = 0
for genes,color,label in groups:
    n = len(genes)
    plt.setp(ax.get_xticklabels()[start:start+n], color=color)
    ax.add_patch(patches.Rectangle((start-0.5,-0.05), n, 0.04, facecolor=color, edgecolor="none", clip_on=False, transform=trans))
    ax.plot([start-0.4,start+n-0.6],[-0.55,-0.55], color="black", lw=1.2, clip_on=False, transform=trans)
    ax.text(start+(n-1)/2,-0.62,label,ha="center",va="top",fontsize=14,clip_on=False,transform=trans)
    start += n

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12); axes[key].set_title(title,fontsize=12)

plt.subplots_adjust(bottom=0.38)
plt.savefig("Fig.4/CD8_apoptosis_pyroptosis_genes_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
score = "Apoptosis scores"
pb_df = CD8T.obs.groupby(["Sample","Condition","celltype"], observed=False)[score].mean().dropna().reset_index()
celltypes = pb_df.groupby("celltype", observed=False)[score].mean().sort_values(ascending=False).index.tolist()

results = []
for ct in celltypes:
    sub = pb_df[pb_df["celltype"]==ct]
    groups = [sub.loc[sub["Condition"]==c,score] for c in ["HC","MKPP","SKPP"]]
    if all(len(g)>=3 for g in groups):
        stat,p = kruskal(*groups); results.append([ct,stat,p])

apo_stats = pd.DataFrame(results, columns=["CellType","H_stat","p_value"])
if not apo_stats.empty:
    apo_stats["p_adj"] = multipletests(apo_stats["p_value"], method="bonferroni")[1]
    apo_stats["Significance"] = apo_stats["p_adj"].apply(lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")

fig,ax = plt.subplots(figsize=(max(6,len(celltypes)*0.6),5))
color = "#E64B35"
sns.boxplot(data=pb_df,x="celltype",y=score,order=celltypes,ax=ax,width=0.6,showfliers=False,
            boxprops={"facecolor":"none","edgecolor":color,"linewidth":1.5},
            whiskerprops={"color":color,"linewidth":1.5},capprops={"color":color,"linewidth":1.5},
            medianprops={"color":color,"linewidth":1.5})

ymin,ymax = pb_df[score].min(),pb_df[score].max(); yrange = ymax-ymin
sig = apo_stats.set_index("CellType")["Significance"] if not apo_stats.empty else {}
for i,ct in enumerate(celltypes):
    if sig.get(ct,"ns")!="ns": ax.text(i,ymax+yrange*0.05,sig[ct],ha="center",va="bottom",fontsize=12,fontweight="bold")

ax.set_ylim(ymin-yrange*0.05,ymax+yrange*0.2)
ax.set_ylabel("Apoptosis Score",fontsize=13,fontweight="bold")
ax.set_xlabel("")
ax.set_xticks(range(len(celltypes)),celltypes,rotation=90,ha="center",fontsize=11)
ax.grid(False); sns.despine(ax=ax)
ax.spines[["left","bottom"]].set_linewidth(1)
ax.tick_params(width=1)

plt.tight_layout()
plt.savefig("Fig.S8/celltype_apoptosis_scores_sorted.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
score_order = ["Exhaustion scores","Cytotoxic scores","Apoptosis scores","Pyroptosis scores"]
condition_order = ["HC","MKPP","SKPP"]

score_matrix = NK.obs.groupby("Condition", observed=False)[score_order].mean().reindex(condition_order).T
score_matrix = score_matrix.apply(lambda x: 2*(x-x.min())/(x.max()-x.min())-1 if x.max()!=x.min() else x*0, axis=1)

cmap = LinearSegmentedColormap.from_list("blue_white_red", ["#4DBBD5B2","white","#E64B35B2"], N=100)
fig, ax = plt.subplots(figsize=(5,2.5))

sns.heatmap(score_matrix, cmap=cmap, vmin=-1, vmax=1, linewidths=0.6, linecolor="white",
            cbar_kws={"label":"Signature score","shrink":0.75,"aspect":15}, ax=ax)

ax.set(xlabel="", ylabel="")
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=15, color="black")
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=15, color="black")

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(colors="black", labelsize=14)
cbar.set_label("Signature score", color="black", fontsize=14)

plt.tight_layout()
plt.savefig("Fig.4/NK_Function_score_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import gseapy as gp
gmt_file = "/home/xiaoquan/scanpy/KP/GSEA/h.all.v2026.1.Hs.symbols.gmt"
CD8T_gsea = CD8_exhaustion_sensitive[CD8_exhaustion_sensitive.obs["Condition"].isin(["HC","MKPP","SKPP"])].copy()
CD8T_gsea.obs["Condition"] = pd.Categorical(CD8T_gsea.obs["Condition"], categories=["HC","MKPP","SKPP"], ordered=True)

def run_gsea(adata, reference):
    label = f"SKPP_vs_{reference}"
    sc.tl.rank_genes_groups(adata, groupby="Condition", groups=["SKPP"], reference=reference, method="wilcoxon", use_raw=False)
    deg = sc.get.rank_genes_groups_df(adata, group="SKPP").dropna(subset=["names","scores"])
    deg.to_csv(f"{outdir}/CD8T_{label}_DEG_wilcoxon.csv", index=False)

    rnk = deg[["names","scores"]].drop_duplicates("names").replace([np.inf,-np.inf],np.nan).dropna().sort_values("scores", ascending=False)
    rnk.to_csv(f"{outdir}/CD8T_{label}.rnk", sep="\t", index=False, header=False)

    res = gp.prerank(rnk=rnk, gene_sets=gmt_file, min_size=15, max_size=500, permutation_num=1000,
                     outdir=None, seed=123, threads=16).res2d
    res.to_csv(f"{outdir}/CD8T_{label}_GSEA_Hallmark.csv", index=False)
    return deg, res.sort_values("NES", ascending=False)

deg_SKPP_HC, gsea_SKPP_HC = run_gsea(CD8T_gsea, "HC")
deg_SKPP_MKPP, gsea_SKPP_MKPP = run_gsea(CD8T_gsea, "MKPP")